In [1]:
import pandas as pd
import pandas as pd
import tensorflow as tf

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

In [6]:
df = pd.read_csv('preprocessed_churn_dataset.csv')

In [7]:
# Separate the target variable (y) from the feature matrix (X)
X = df.drop(columns=['Churn'])
y = df['Churn']

In [8]:
# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [9]:
input_dim = X_train.shape[1]

model = tf.keras.Sequential([
    # Input layer and first hidden layer with Batch Normalization and Dropout
    tf.keras.layers.Dense(64, activation='relu', input_shape=(input_dim,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    
    # Second hidden layer
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    
    # Output layer for binary classification
    tf.keras.layers.Dense(1, activation='sigmoid')
])

c:\Users\User\ODL_Labs\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
# Compile model using the Adam optimizer and Binary Crossentropy loss
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

In [11]:
# Define callbacks for training efficiency and optimization
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', 
        patience=5, 
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', 
        factor=0.2, 
        patience=3, 
        min_lr=1e-6
    )
]

In [12]:
# Fit the neural network model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=512,  # Optimized for large-scale tabular datasets (~500k rows)
    callbacks=callbacks,
    verbose=1
)

Epoch 1/30
790/790 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.8788 - auc: 0.9247 - loss: 0.3303 - val_accuracy: 0.9108 - val_auc: 0.9428 - val_loss: 0.2582 - learning_rate: 0.0010
Epoch 2/30
790/790 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9171 - auc: 0.9419 - loss: 0.2525 - val_accuracy: 0.9283 - val_auc: 0.9474 - val_loss: 0.2250 - learning_rate: 0.0010
Epoch 3/30
790/790 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9272 - auc: 0.9451 - loss: 0.2289 - val_accuracy: 0.9310 - val_auc: 0.9491 - val_loss: 0.2122 - learning_rate: 0.0010
Epoch 4/30
790/790 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9298 - auc: 0.9464 - loss: 0.2211 - val_accuracy: 0.9300 - val_auc: 0.9497 - val_loss: 0.2112 - learning_rate: 0.0010
Epoch 5/30
790/790 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9306 - auc: 0.9477 - loss: 0.2168 - val_accuracy: 0.9316 - val_auc: 0.9504 - val_loss: 0.2065 - learning_rate: 0.0010
Epoch 6/30
790/790 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9309 - auc: 0.

In [13]:
# Evaluate model performance on the validation partition
val_loss, val_accuracy, val_auc = model.evaluate(X_val, y_val, verbose=0)
print("\n=== Validation Set Metrics ===")
print(f"Loss:     {val_loss:.4f}")
print(f"Accuracy: {val_accuracy:.4f}")
print(f"AUC-ROC:  {val_auc:.4f}\n")


=== Validation Set Metrics ===
Loss:     0.1927
Accuracy: 0.9328
AUC-ROC:  0.9527



In [14]:
# Generate probability predictions and class classifications
y_pred_prob = model.predict(X_val, verbose=0)
y_pred_class = (y_pred_prob > 0.5).astype(int)

In [15]:
# Display comprehensive classification report
print("=== Classification Report ===")
print(classification_report(y_val, y_pred_class))

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.99      0.86      0.92     44943
           1       0.90      0.99      0.94     56099

    accuracy                           0.93    101042
   macro avg       0.94      0.93      0.93    101042
weighted avg       0.94      0.93      0.93    101042

